# Projeto LAUV — Modelagem e Controle

Modelagem dinâmica e projeto de controladores para o veículo submarino autônomo leve (LAUV),
nos planos **vertical** (controle de cota por alocação de polos) e **horizontal** (controle de rumo proporcional).

## 1. Configuração

In [ ]:
import math
import warnings

import control
import matplotlib.pyplot as plt
import mplcyberpunk
import numpy as np
import sympy as sp
from control import ctrb, tf
from scipy.integrate import solve_ivp
from scipy.signal import StateSpace, lsim
from sympy import Matrix, Poly, eye, factor, pprint, simplify, solve, symbols

In [ ]:
laranja = '#ff7f0e'
azul = '#1f77b4'

ciano_neon = '#00FFFF'
laranja_neon = '#FF5F1F'
amarelo_acido = '#FFFF33'
vermelho_neon = '#FF073A'
magenta = '#FF00FF'
verde_limao = '#39FF14'
roxo_eletrico = '#BF00FF'

## 2. Parâmetros do Sistema

Parâmetros físicos e geométricos do LAUV e de seus apêndices (lemes e casco).
Definidos uma única vez e reutilizados em todas as seções.

In [ ]:
rho = 1025  # massa específica da água (kg/m³)
CDl = 0.01  # coeficiente de arrasto dos lemes — fólio fino (adimensional)
CDc = 1.0  # coeficiente de arrasto 2D do casco — cilindro (adimensional)
c = 0.026  # corda do fólio (m)
b = 0.052  # envergadura do fólio (m)
m = 16.0  # massa do AUV (kg)
D = 0.15  # diâmetro do AUV (m)
L = 1.2  # comprimento do AUV (m)
GB = D / 4  # distância entre G e B (m)
Jyy = (
    (m / 2) * (1 / 12) * (3 * ((D / 2) ** 2) + (L**2))
    + (m / 2) * ((D / 2) ** 2)
)  # momento de inércia de pitch em relação a B (kg·m²)
Jzz = m * (1 / 12) * (3 * ((D / 2) ** 2) + (L**2))  # momento de inércia de yaw (kg·m²)
g = 9.81  # aceleração gravitacional (m/s²)
d = 0.9 * (L / 2)  # distância do fólio à meia nau (m)
U = 2.0  # velocidade horizontal de equilíbrio (m/s)
m11 = 0.029 * rho * math.pi * (D**2) * L * (1 / 6)  # massa adicional de surge (kg)
m33 = 0.96 * rho * math.pi * (D**2) * L * (1 / 4)  # massa adicional de heave (kg)
m22 = m33  # massa adicional de sway (kg)
m55 = 0.96 * rho * math.pi * (L**3) * (1 / 12) * (D**2)  # inércia adicional de pitch (kg·m²)
m66 = m55  # inércia adicional de yaw (kg·m²)
omega_m = math.radians(10) / 1  # velocidade angular máxima (rad/s)
theta_m = math.radians(15)  # ângulo máximo de pitch para linearização (rad)
defl = -10  # deflexão do leme no teste degrau (graus)
ta = 10  # início da atuação do leme (s)
tb = 40  # fim da atuação do leme (s)

## 3. Modelagem

### 3.1 Funções Auxiliares

Forças hidrodinâmicas nos fólios e no casco; deflexão de leme em degrau.

In [ ]:
def Lift_foil(u, w, alpha):
    CL = 2 * math.pi * np.sin(alpha)
    return (1 / 2) * rho * c * (u**2 + w**2) * b * CL


def Drag_foil(u, w):
    return (1 / 2) * rho * c * CDl * (u**2 + w**2) * b


def Lift_hull(u, w, beta):
    CL0 = 2 * math.pi * np.sin(beta)
    CL = CL0 / (1 + (CL0 / (math.pi * ((D**2) / D * L) * 0.8)))  # correção 3D
    return (1 / 2) * rho * L * (u**2 + w**2) * D * CL


def phi_foil(t):
    if ta <= t <= tb:
        return math.radians(defl)
    return 0

### 3.2 Equações de Movimento Não Lineares

Sistema de EDOs para os planos vertical ($u, w, \theta$) e horizontal ($u, v, \psi$),
obtido pela segunda lei de Newton na forma matricial e resolvido a cada passo de tempo.

In [ ]:
def LAUVv_non_linear(t, X):  # plano vertical
    u, w, theta, theta_dot = X

    beta = np.arctan2(w, u)
    phi = phi_foil(t)
    alpha = phi - beta

    Lf = Lift_foil(u, w, alpha)
    Rf = Drag_foil(u, w)
    Lc = Lift_hull(u, w, beta)

    A = np.array(
        [
            [m, 0, -m * GB],
            [0, m + m33, 0],
            [-GB * m, 0, Jyy + m55],
        ]
    )
    b = np.array(
        [
            -m * w * theta_dot,
            (
                2 * Lf * np.cos(beta)
                - 2 * Rf * np.sin(beta)
                + (m + m11) * u * theta_dot
                - m * GB * theta_dot**2
                - (1 / 2) * rho * 0.82 * CDc * D * L * w * np.abs(w)
                - Lc * np.cos(beta)
            ),
            (
                -(m * g * GB) * np.sin(theta)
                + 2 * (Lf * np.cos(beta) - Rf * np.sin(beta)) * d
                + m * GB * w * theta_dot
                - 50
                * (1 / 32)
                * rho
                * 0.80
                * CDc
                * D
                * (L**4)
                * theta_dot
                * np.abs(theta_dot)
                + Lc * (L / 4) * np.cos(beta)
            ),
        ]
    )

    du_dt, dw_dt, dtheta_dot_dt = np.linalg.solve(A, b)
    return [du_dt, dw_dt, theta_dot, dtheta_dot_dt]


def LAUVh_non_linear(t, X):  # plano horizontal
    u, v, psi, psi_dot = X

    beta = np.arctan2(v, u)
    phi = phi_foil(t)
    alpha = phi - beta

    Lf = Lift_foil(u, v, alpha)
    Rf = Drag_foil(u, v)
    Lc = Lift_hull(u, v, beta)

    A = np.array(
        [
            [m, 0, 0],
            [0, m + m22, 0],
            [0, 0, Jzz + m66],
        ]
    )
    b = np.array(
        [
            m * v * psi_dot,
            (
                2 * Lf * np.cos(beta)
                - 2 * Rf * np.sin(beta)
                - (m + m11) * u * psi_dot
                - (1 / 2) * rho * 0.82 * CDc * D * L * v * np.abs(v)
                - Lc * np.cos(beta)
            ),
            (
                -2 * (Lf * np.cos(beta) - Rf * np.sin(beta)) * d
                - 50
                * (1 / 32)
                * rho
                * 0.80
                * CDc
                * D
                * (L**4)
                * psi_dot
                * np.abs(psi_dot)
                - Lc * (L / 4) * np.cos(beta)
            ),
        ]
    )

    du_dt, dw_dt, dpsi_dot_dt = np.linalg.solve(A, b)
    return [du_dt, dw_dt, psi_dot, dpsi_dot_dt]

### 3.3 Sistema Linear — Espaço de Estados

Matrizes $A$, $B$, $C$ obtidas pela linearização em torno do ponto de equilíbrio ($U = 2$ m/s, $\theta = 0$).

In [ ]:
# --- Plano Vertical ---
Av = np.zeros((4, 4))
Bv = np.zeros((4, 1))
Cv = np.zeros((3, 4))
Ddv = np.zeros((3, 1))

Av[0, 1] = GB * (
    -2 * rho * c * b * U * math.pi * d
    - 2 * 0.5 * rho * CDl * c * b * U * d
    + 0.25 * rho * (L**2) * D * U * math.pi
) / (Jyy + m55 - m * (GB**2))
Av[0, 2] = -(m * g * (GB**2)) / (Jyy + m55 - m * (GB**2))
Av[0, 3] = (
    -(omega_m * GB * rho * 0.80 * CDc * D * (L**4))
    / (Jyy + m55 - m * (GB**2))
    * (50 * (3 / 4) * (1 / 32))
)

Av[1, 1] = -(
    (3 / 8) * U * math.sin(theta_m) * rho * 0.82 * CDc * D * L
    + 2 * rho * c * U * math.pi * b
    + 2 * 0.5 * rho * CDl * c * b * U
    + rho * L * D * U * math.pi
) / (m + m33)
Av[1, 3] = U * (m + m11) / (m + m33)

Av[2, 3] = 1

Av[3, 1] = (
    -2 * rho * c * b * U * math.pi * d
    - 2 * 0.5 * rho * CDl * c * b * U * d
    + 0.25 * rho * (L**2) * D * U * math.pi
) / (Jyy + m55 - m * (GB**2))
Av[3, 2] = -(m * g * GB) / (Jyy + m55 - m * (GB**2))
Av[3, 3] = (
    -(omega_m * rho * 0.80 * CDc * D * (L**4))
    / (Jyy + m55 - m * (GB**2))
    * (50 * (3 / 4) * (1 / 32))
)

Bv[0, 0] = (2 * rho * c * b * (U**2) * math.pi * d * GB) / (Jyy + m55 - m * (GB**2))
Bv[1, 0] = (2 * rho * c * (U**2) * math.pi * b) / (m + m33)
Bv[3, 0] = (2 * rho * c * b * (U**2) * math.pi * d) / (Jyy + m55 - m * (GB**2))

Cv[0, 0] = 1
Cv[1, 1] = 1
Cv[2, 2] = 1

LAUVv_linear = StateSpace(Av, Bv, Cv, Ddv)

# --- Plano Horizontal ---
Ah = np.zeros((4, 4))
Bh = np.zeros((4, 1))
Ch = np.zeros((3, 4))
Ddh = np.zeros((3, 1))

Ah[1, 1] = -(
    (3 / 8) * U * math.sin(theta_m) * rho * 0.82 * CDc * D * L
    + 2 * rho * c * U * math.pi * b
    + 2 * 0.5 * rho * CDl * c * b * U
    + rho * L * D * U * math.pi
) / (m + m22)
Ah[1, 3] = -U * (m + m11) / (m + m22)

Ah[2, 3] = 1

Ah[3, 1] = (
    2 * rho * c * b * U * math.pi * d
    + 2 * 0.5 * rho * CDl * c * b * U * d
    - 0.25 * rho * (L**2) * D * U * math.pi
) / (Jzz + m66)
Ah[3, 3] = (
    -(omega_m * rho * 0.80 * CDc * D * (L**4))
    / (Jzz + m66)
    * (50 * (3 / 4) * (1 / 32))
)

Bh[1, 0] = (2 * rho * c * (U**2) * math.pi * b) / (m + m33)
Bh[3, 0] = -(2 * rho * c * b * (U**2) * math.pi * d) / (Jzz + m66)

Ch[0, 0] = 1
Ch[1, 1] = 1
Ch[2, 2] = 1

LAUVh_linear = StateSpace(Ah, Bh, Ch, Ddh)

### 3.4 Simulação — Resposta ao Degrau de Leme

Integração numérica (RK45) do modelo não linear e simulação do modelo linear
para uma deflexão de leme em degrau de $\phi = -10°$ no intervalo $[10, 40]$ s.

In [ ]:
x0 = [U, 0.0, 0.0, 0.0]
t_span = (0, 80)
t_eval = np.linspace(t_span[0], t_span[-1], 1000)

phi = math.radians(defl) * ((t_eval >= ta) & (t_eval <= tb)).astype(float)

sol_v = solve_ivp(LAUVv_non_linear, t_span, x0, t_eval=t_eval, method='RK45')
sol_h = solve_ivp(LAUVh_non_linear, t_span, x0, t_eval=t_eval, method='RK45')
tv_out, yv_out, xv_out = lsim(LAUVv_linear, U=phi, T=t_eval, X0=x0)
th_out, yh_out, xh_out = lsim(LAUVh_linear, U=phi, T=t_eval, X0=x0)

# Plano vertical
u_non_lin_v = sol_v.y[0]
w_non_lin = sol_v.y[1]
theta_non_lin = sol_v.y[2]

u_lin_v = yv_out[:, 0]
w_lin = yv_out[:, 1]
theta_lin = yv_out[:, 2]

# Plano horizontal
u_non_lin_h = sol_h.y[0]
v_non_lin = sol_h.y[1]
psi_non_lin = sol_h.y[2]

u_lin_h = yh_out[:, 0]
v_lin = yh_out[:, 1]
psi_lin = yh_out[:, 2]

# Conversão para base fixa — plano vertical
U_non_lin_v = u_non_lin_v * np.cos(theta_non_lin) + w_non_lin * np.sin(theta_non_lin)
W_non_lin = -u_non_lin_v * np.sin(theta_non_lin) + w_non_lin * np.cos(theta_non_lin)

U_lin_v = u_lin_v * np.cos(theta_lin) + w_lin * np.sin(theta_lin)
W_lin = -u_lin_v * np.sin(theta_lin) + w_lin * np.cos(theta_lin)

# Conversão para base fixa — plano horizontal
U_non_lin_h = u_non_lin_h * np.cos(psi_non_lin) - v_non_lin * np.sin(psi_non_lin)
V_non_lin = u_non_lin_h * np.sin(psi_non_lin) + v_non_lin * np.cos(psi_non_lin)

U_lin_h = u_lin_h * np.cos(psi_lin) - v_lin * np.sin(psi_lin)
V_lin = u_lin_h * np.sin(psi_lin) + v_lin * np.cos(psi_lin)

# Integração de posições
X0 = 0
Y0 = 0
Z0 = -100
dt = sol_v.t[1] - sol_v.t[0]

X_non_lin_v = X0 + np.cumsum(U_non_lin_v) * dt
Z_non_lin = Z0 + np.cumsum(W_non_lin) * dt

X_lin_v = X0 + np.cumsum(U_lin_v) * dt
Z_lin = Z0 + np.cumsum(W_lin) * dt

X_non_lin_h = X0 + np.cumsum(U_non_lin_h) * dt
Y_non_lin = Y0 + np.cumsum(V_non_lin) * dt

X_lin_h = X0 + np.cumsum(U_lin_h) * dt
Y_lin = Y0 + np.cumsum(V_lin) * dt

### 3.5 Resultados — Teste de Leme

In [ ]:
# Trajetória — plano vertical
plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 2)
plt.plot(X_non_lin_v, Z_non_lin, color=verde_limao, linestyle='-', label='Modelo Não Linear')
plt.plot(X_lin_v, Z_lin, color=verde_limao, linestyle='--', label='Modelo Linear')
plt.legend(loc='center right')
plt.xlabel('Posição X (m)')
plt.ylabel('Posição Z (m)')
plt.grid(True)
plt.title(
    f'Trajetória do LAUV no Plano Vertical — Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s',
    fontsize=14,
)
plt.tight_layout()
plt.show()

# Trajetória — plano horizontal
plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 2)
plt.plot(X_non_lin_h, Y_non_lin, color=verde_limao, linestyle='-', label='Modelo Não Linear')
plt.plot(X_lin_h, Y_lin, color=verde_limao, linestyle='--', label='Modelo Linear')
plt.legend(loc='lower right')
plt.xlabel('Posição X (m)')
plt.ylabel('Posição Y (m)')
plt.grid(True)
plt.title(
    f'Trajetória do LAUV no Plano Horizontal — Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s',
    fontsize=14,
)
plt.tight_layout()
plt.show()

# Velocidades — plano vertical
plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 2)
plt.plot(sol_v.t, U_non_lin_v, color=ciano_neon, linestyle='-', label='surge (modelo não linear)')
plt.plot(sol_v.t, U_lin_v, color=ciano_neon, linestyle='--', label='surge (modelo linear)')
plt.plot(sol_v.t, W_non_lin, color=laranja_neon, linestyle='-', label='heave (modelo não linear)')
plt.plot(sol_v.t, W_lin, color=laranja_neon, linestyle='--', label='heave (modelo linear)')
plt.legend(loc='center right')
plt.xlabel('Tempo (s)')
plt.ylabel('Velocidades (m/s)')
plt.grid(True)
plt.title(
    f'Velocidades do LAUV no Plano Vertical — Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s',
    fontsize=14,
)
plt.tight_layout()
plt.show()

# Velocidades — plano horizontal
plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 2)
plt.plot(sol_h.t, U_non_lin_h, color=ciano_neon, linestyle='-', label='surge (modelo não linear)')
plt.plot(sol_h.t, U_lin_h, color=ciano_neon, linestyle='--', label='surge (modelo linear)')
plt.plot(sol_h.t, V_non_lin, color=laranja_neon, linestyle='-', label='sway (modelo não linear)')
plt.plot(sol_h.t, V_lin, color=laranja_neon, linestyle='--', label='sway (modelo linear)')
plt.legend(loc='upper right')
plt.xlabel('Tempo (s)')
plt.ylabel('Velocidades (m/s)')
plt.grid(True)
plt.title(
    f'Velocidades do LAUV no Plano Horizontal — Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s',
    fontsize=14,
)
plt.tight_layout()
plt.show()

# Ângulo de pitch
plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 2)
plt.plot(
    sol_v.t, np.rad2deg(theta_non_lin), color=vermelho_neon, linestyle='-',
    label=r'$\theta$(t) — Modelo Não Linear',
)
plt.plot(
    sol_v.t, np.rad2deg(theta_lin), color=vermelho_neon, linestyle='--',
    label=r'$\theta$(t) — Modelo Linear',
)
plt.legend(loc='center right')
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (graus)')
plt.grid(True)
plt.title(
    f'Ângulo de Pitch do LAUV — Plano Vertical — Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s',
    fontsize=14,
)
plt.tight_layout()
plt.show()

# Ângulo de yaw
plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 2)
plt.plot(
    sol_h.t, np.rad2deg(psi_non_lin), color=vermelho_neon, linestyle='-',
    label=r'$\psi$(t) — Modelo Não Linear',
)
plt.plot(
    sol_v.t, np.rad2deg(psi_lin), color=vermelho_neon, linestyle='--',
    label=r'$\psi$(t) — Modelo Linear',
)
plt.legend(loc='center right')
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (graus)')
plt.grid(True)
plt.title(
    f'Ângulo de Yaw do LAUV — Plano Horizontal — Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s',
    fontsize=14,
)
plt.tight_layout()
plt.show()

## 4. Controle — Plano Vertical

Controlador de cota ($z$) por realimentação de estado com alocação de polos.
O vetor de estados controlável é $\mathbf{x} = [w,\, \theta,\, \dot{\theta},\, z]^\top$.

### 4.1 Condições de Simulação

In [ ]:
Time = [0, 80]
Uc = 2.0
X0 = np.array([Uc, 0, 0, 0])  # [u, w, theta, theta_dot]
z0 = -100.0  # posição vertical inicial (m)
z_ref = -80.0  # cota de referência (m)

phi_max = np.deg2rad(10.0)  # saturação do atuador (rad)
pitch_max = np.deg2rad(15)  # ângulo máximo de pitch (rad)

### 4.2 Análise de Controlabilidade

Verificação da controlabilidade do sistema original e construção do sistema
em forma controlável com polo integrador de posição vertical.

In [ ]:
CO = ctrb(Av, Bv)
print(f'\nMatriz de controlabilidade CO =\n{CO}')
print(f'\nPosto de CO: {np.linalg.matrix_rank(CO)}')

# Sistema controlável — estados: [w, theta, theta_dot, z]
Avc = np.zeros((4, 4))
Bvc = np.zeros((4, 1))

Avc[0, 0] = -(
    (3 / 8) * U * math.sin(theta_m) * rho * 0.82 * CDc * D * L
    + 2 * rho * c * U * math.pi * b
    + 2 * 0.5 * rho * CDl * c * b * U
    + rho * L * D * U * math.pi
) / (m + m33)
Avc[0, 2] = U * (m + m11) / (m + m33)

Avc[1, 2] = 1

Avc[2, 0] = (
    -2 * rho * c * b * U * math.pi * d
    - 2 * 0.5 * rho * CDl * c * b * U * d
    + 0.25 * rho * (L**2) * D * U * math.pi
) / (Jyy + m55 - m * (GB**2))
Avc[2, 1] = -(m * g * GB) / (Jyy + m55 - m * (GB**2))
Avc[2, 2] = (
    -(omega_m * rho * 0.80 * CDc * D * (L**4))
    / (Jyy + m55 - m * (GB**2))
    * (100 * (3 / 4) * (1 / 64))
)

Avc[3, 0] = 1
Avc[3, 1] = -U

Bvc[0, 0] = (2 * rho * c * (U**2) * math.pi * b) / (m + m33)
Bvc[2, 0] = (2 * rho * c * b * (U**2) * math.pi * d) / (Jyy + m55 - m * (GB**2))

COc = ctrb(Avc, Bvc)
print(f'\nMatriz de controlabilidade COc =\n{COc}')
print(f'\nPosto de COc: {np.linalg.matrix_rank(COc)}')

print(f'\nMatriz Av =\n{Av}')
print(f'\nMatriz Bv =\n{Bv}')
print(f'\nMatriz Avc =\n{Avc}')
print(f'\nMatriz Bvc =\n{Bvc}')

eigvals_v, eigvecs_v = np.linalg.eig(Av)
eigvals_vc, eigvecs_vc = np.linalg.eig(Avc)
print(f'\nAutovalores de Av: {eigvals_v}')
print(f'\nAutovalores de Avc: {eigvals_vc}')

### 4.3 Projeto do Controlador por Alocação de Polos

Ganho de realimentação $K$ calculado pelo método de alocação de polos.
Pré-compensador $k_r$ para rastreamento de referência sem erro em regime permanente.

In [ ]:
wn = 1.8
zeta = 0.707
s1 = -wn * zeta + 1j * wn * np.sqrt(1 - zeta**0.5)
s2 = -wn * zeta - 1j * wn * np.sqrt(1 - zeta**0.5)
s3 = 5 * (-wn * zeta)
s4 = 6 * (-wn * zeta)
polos = [s1, s2, s3, s4]

Kcon = control.place(Avc, Bvc, polos)

print(f'\nPolos alocados: {polos}')
print(f'\nGanho de realimentação K = {Kcon}')

# Pré-compensador kr
Av_con = Avc - Bvc @ Kcon
C = np.array([0, 0, 0, 1])

temp = np.linalg.solve(Av_con, Bvc)
denom = (C @ temp).item()

if abs(denom) < 1e-9:
    raise RuntimeError(
        'Denominador próximo de zero: (C (A-BK)^{-1} B) ≈ 0. '
        'Não é possível calcular kr diretamente. Considere integrar erro.'
    )
kr = -1.0 / denom

print(f'\nGanho de referência kr = {kr}\n')

### 4.4 Funções de Dinâmica em Malha Fechada

Quatro variantes: modelos linear e não linear, com e sem saturação de pitch.

In [ ]:
def LAUVvc_lin_psat(t, X):
    X = X.reshape(-1, 1)

    X[1] = np.clip(X[1], -pitch_max, pitch_max)

    u = float(-(Kcon @ X) + kr * z_ref)
    u_sat = np.clip(u, -phi_max, phi_max)

    X_dot = Avc @ X + Bvc * u_sat
    return X_dot.flatten()


def LAUVvc_lin(t, X):
    X = X.reshape(-1, 1)

    u = float(-(Kcon @ X) + kr * z_ref)
    u_sat = np.clip(u, -phi_max, phi_max)

    X_dot = Avc @ X + Bvc * u_sat
    return X_dot.flatten()


def LAUVvc_non_lin_psat(t, X_aug):
    u, w, theta, theta_dot, z = X_aug
    X = np.hstack((w, theta, theta_dot, z))

    theta = np.clip(theta, -pitch_max, pitch_max)
    X_aug[2] = theta
    X[1] = theta

    W = -u * np.sin(theta) + w * np.cos(theta)

    u_ctrl = float(-(Kcon @ X) + kr * z_ref)
    u_sat = np.clip(u_ctrl, -phi_max, phi_max)

    beta = np.arctan2(w, u)
    alpha = u_sat - beta

    Lf = Lift_foil(u, w, alpha)
    Rf = Drag_foil(u, w)
    Lc = Lift_hull(u, w, beta)

    A = np.array(
        [
            [m, 0, -m * GB],
            [0, m + m33, 0],
            [-GB * m, 0, Jyy + m55],
        ]
    )
    b = np.array(
        [
            -m * w * theta_dot,
            (
                2 * Lf * np.cos(beta)
                - 2 * Rf * np.sin(beta)
                + (m + m11) * u * theta_dot
                - m * GB * theta_dot**2
                - (1 / 2) * rho * 0.82 * CDc * D * L * w * np.abs(w)
                - Lc * np.cos(beta)
            ),
            (
                -(m * g * GB) * np.sin(theta)
                + 2 * (Lf * np.cos(beta) - Rf * np.sin(beta)) * d
                + m * GB * w * theta_dot
                - 100
                * (1 / 64)
                * rho
                * 0.80
                * CDc
                * D
                * (L**4)
                * theta_dot
                * np.abs(theta_dot)
                + Lc * (L / 4) * np.cos(beta)
            ),
        ]
    )

    du_dt, dw_dt, dtheta_dot_dt = np.linalg.solve(A, b)
    dtheta_dt = theta_dot
    z_dot = W

    return np.hstack((du_dt, dw_dt, dtheta_dt, dtheta_dot_dt, z_dot))


def LAUVvc_non_lin(t, X_aug):
    u, w, theta, theta_dot, z = X_aug
    X = np.hstack((w, theta, theta_dot, z))

    W = -u * np.sin(theta) + w * np.cos(theta)

    u_ctrl = float(-(Kcon @ X) + kr * z_ref)
    u_sat = np.clip(u_ctrl, -phi_max, phi_max)

    beta = np.arctan2(w, u)
    alpha = u_sat - beta

    Lf = Lift_foil(u, w, alpha)
    Rf = Drag_foil(u, w)
    Lc = Lift_hull(u, w, beta)

    A = np.array(
        [
            [m, 0, -m * GB],
            [0, m + m33, 0],
            [-GB * m, 0, Jyy + m55],
        ]
    )
    b = np.array(
        [
            -m * w * theta_dot,
            (
                2 * Lf * np.cos(beta)
                - 2 * Rf * np.sin(beta)
                + (m + m11) * u * theta_dot
                - m * GB * theta_dot**2
                - (1 / 2) * rho * 0.82 * CDc * D * L * w * np.abs(w)
                - Lc * np.cos(beta)
            ),
            (
                -(m * g * GB) * np.sin(theta)
                + 2 * (Lf * np.cos(beta) - Rf * np.sin(beta)) * d
                + m * GB * w * theta_dot
                - 100
                * (1 / 64)
                * rho
                * 0.80
                * CDc
                * D
                * (L**4)
                * theta_dot
                * np.abs(theta_dot)
                + Lc * (L / 4) * np.cos(beta)
            ),
        ]
    )

    du_dt, dw_dt, dtheta_dot_dt = np.linalg.solve(A, b)
    dtheta_dt = theta_dot
    z_dot = W

    return np.hstack((du_dt, dw_dt, dtheta_dt, dtheta_dot_dt, z_dot))

### 4.5 Simulação

In [ ]:
X0c = [0, 0, 0, z0]  # [w, theta, theta_dot, z]
t_span = (Time[0], Time[1])
t_eval = np.linspace(*t_span, 2000)

solc_lin_psat = solve_ivp(LAUVvc_lin_psat, t_span, X0c, t_eval=t_eval)
solc_lin = solve_ivp(LAUVvc_lin, t_span, X0c, t_eval=t_eval)

zc_lin_psat = solc_lin_psat.y[3]
thetac_lin_psat = solc_lin_psat.y[1]

zc_lin = solc_lin.y[3]
thetac_lin = solc_lin.y[1]

X0_aug = np.hstack((X0, [z0]))

solv_non_lin_psat = solve_ivp(
    LAUVvc_non_lin_psat, t_span, X0_aug, t_eval=t_eval, method='RK45'
)
solv_non_lin = solve_ivp(
    LAUVvc_non_lin, t_span, X0_aug, t_eval=t_eval, method='RK45'
)

u_non_lin_psat = solv_non_lin_psat.y[0]
w_non_lin_psat = solv_non_lin_psat.y[1]
theta_non_lin_psat = solv_non_lin_psat.y[2]
z_non_lin_psat = solv_non_lin_psat.y[4]

u_non_lin = solv_non_lin.y[0]
w_non_lin = solv_non_lin.y[1]
theta_non_lin = solv_non_lin.y[2]
z_non_lin = solv_non_lin.y[4]

# Atuador
u_psat = np.zeros(solc_lin_psat.t.size)
for i in range(solc_lin_psat.t.size):
    X = solc_lin_psat.y[:, i]
    u_ctrl = float(-(Kcon @ X) + kr * z_ref)
    u_psat[i] = np.clip(u_ctrl, -phi_max, phi_max)

u_ctrl_arr = np.zeros(solc_lin.t.size)
for i in range(solc_lin.t.size):
    X = solc_lin.y[:, i]
    u_ctrl = float(-(Kcon @ X) + kr * z_ref)
    u_ctrl_arr[i] = np.clip(u_ctrl, -phi_max, phi_max)

### 4.6 Resultados — Controle Vertical

In [ ]:
# --- Com saturação de pitch ---

# Posição vertical
plt.figure(figsize=(10, 6))
plt.text(
    0.76, 0.40,
    f'$K_r$ = {kr:.2f}',
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4'),
)
plt.plot(
    solc_lin_psat.t, zc_lin_psat, color=ciano_neon, linestyle='--',
    label='Z(t): Posição Vertical do LAUV — Modelo Linear',
)
plt.plot(
    solv_non_lin_psat.t, z_non_lin_psat, color=ciano_neon, linestyle='-',
    label='Z(t): Posição Vertical do LAUV — Modelo Não Linear',
)
plt.axhline(z_ref, color='r', linestyle='--', label=f'cota de referência: Z = {z_ref:.1f} m')
plt.xlabel('Tempo (s)')
plt.ylabel('Posição Z (m)')
plt.title('Controlador de Cota Vertical do LAUV com Restrição de Pitch — Plano Vertical')
plt.legend(loc='center right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Ângulo de pitch
plt.figure(figsize=(10, 6))
plt.text(
    0.75, 0.62,
    rf'$\theta_{{máx}}$ = {np.rad2deg(pitch_max):.1f} graus',
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4'),
)
plt.plot(
    solc_lin_psat.t, np.rad2deg(thetac_lin_psat), color=amarelo_acido, linestyle='--',
    label=r'$\theta$(t): Orientação Angular do LAUV — Modelo Linear',
)
plt.plot(
    solv_non_lin_psat.t, np.rad2deg(theta_non_lin_psat), color=amarelo_acido, linestyle='-',
    label=r'$\theta$(t): Orientação Angular do LAUV — Modelo Não Linear',
)
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (Graus)')
plt.title('Ângulo de Pitch com Restrição — Controle no Espaço de Estado')
plt.legend(loc='center right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Atuador
plt.figure(figsize=(10, 6))
plt.text(
    0.83, 0.90,
    f'$\phi_{{máx}}$ = {np.rad2deg(phi_max):.1f} graus',
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4'),
)
plt.plot(solc_lin_psat.t, np.rad2deg(u_psat), color=laranja_neon, linestyle='-', label='$\phi$(t): Ângulo do Leme')
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (graus)')
plt.title('Ângulo de Deflexão do Leme Saturado — Pitch com Restrição')
plt.legend(loc='upper right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# --- Sem saturação de pitch ---

# Posição vertical
plt.figure(figsize=(10, 6))
plt.text(
    0.76, 0.40,
    f'$K_r$ = {kr:.2f}',
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4'),
)
plt.plot(
    solc_lin.t, zc_lin, color=ciano_neon, linestyle='--',
    label='Z(t): Posição Vertical do LAUV — Modelo Linear',
)
plt.plot(
    solv_non_lin.t, z_non_lin, color=ciano_neon, linestyle='-',
    label='Z(t): Posição Vertical do LAUV — Modelo Não Linear',
)
plt.axhline(z_ref, color='r', linestyle='--', label=f'cota de referência: Z = {z_ref:.1f} m')
plt.xlabel('Tempo (s)')
plt.ylabel('Posição Z (m)')
plt.title('Controlador de Cota Vertical do LAUV sem Restrição de Pitch — Plano Vertical')
plt.legend(loc='center right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Ângulo de pitch
plt.figure(figsize=(10, 6))
plt.plot(
    solc_lin.t, np.rad2deg(thetac_lin), color=amarelo_acido, linestyle='--',
    label=r'$\theta$(t): Orientação Angular do LAUV — Modelo Linear',
)
plt.plot(
    solv_non_lin.t, np.rad2deg(theta_non_lin), color=amarelo_acido, linestyle='-',
    label=r'$\theta$(t): Orientação Angular do LAUV — Modelo Não Linear',
)
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (Graus)')
plt.title('Ângulo de Pitch sem Restrição — Controle no Espaço de Estado')
plt.legend(loc='center right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Atuador
plt.figure(figsize=(10, 6))
plt.text(
    0.83, 0.90,
    f'$\phi_{{máx}}$ = {np.rad2deg(phi_max):.1f} graus',
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4'),
)
plt.plot(solc_lin.t, np.rad2deg(u_ctrl_arr), color=laranja_neon, linestyle='-', label='$\phi$(t): Ângulo do Leme')
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (graus)')
plt.title('Ângulo de Deflexão do Leme Saturado — Pitch sem Restrição')
plt.legend(loc='upper right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

## 5. Controle — Plano Horizontal

Controlador de rumo ($\psi$) proporcional com análise via lugar das raízes.
O vetor de estados controlável é $\mathbf{x} = [v,\, \psi,\, \dot{\psi}]^\top$.

### 5.1 Função de Transferência e Autovalores

Derivação simbólica da função de transferência $G(s) = \psi(s)/\phi(s)$
e verificação dos autovalores do sistema em malha aberta.

In [ ]:
eigvals, eigvecs = np.linalg.eig(Ah)
print('\nAutovalores de Ah:', eigvals)

print(f'\nMatriz Ah =\n{Ah}')
print(f'\nMatriz Bh =\n{Bh}')

R = Ah[3][3] + Ah[1][1]
Q = Ah[3][1] * Ah[1][3] - Ah[3][3] * Ah[1][1]

print(f'\nCoeficientes do polinômio característico: a3 = {-1}; a2 = {R:.1f}; a1 = {Q:.1f}; a0 = {0}')
print(
    f'\nCoeficientes do numerador: '
    f'a1 = {-Bh[3][0]:.1f}; '
    f'a0 = {(Bh[3][0] * Ah[1][1] - Ah[3][1] * Bh[1][0]):.1f}'
)

# Derivação simbólica da função de transferência
s = symbols('s')
ai, bi, ci, di, x, y = symbols('a b c d x y')

A_sym = Matrix(
    [
        [0, 0, 0, 0],
        [0, ci, 0, di],
        [0, 0, 0, 1],
        [0, ai, 0, bi],
    ]
)
B_sym = Matrix([[0], [x], [0], [y]])
C_sym = Matrix([[0, 0, 1, 0]])
Dd_sym = Matrix([[0]])

sI_minus_A = s * eye(A_sym.rows) - A_sym
inv = sI_minus_A.inv()
G = simplify(C_sym * inv * B_sym + Dd_sym)
G_factored = factor(G)

print('\nG(s) fatorado:')
pprint(G_factored)

# Raízes numéricas para comparação
s_sym = symbols('s')
p = -s_sym**3 + R * s_sym**2 + Q * s_sym
poly = Poly(p, s_sym)
roots_poly = poly.nroots()
print('\nRaízes numéricas da equação característica:', roots_poly, '\n')

### 5.2 Lugar das Raízes

Cálculo numérico do lugar das raízes para o controlador proporcional de rumo.
Faixa de estabilidade determinada pelo critério de Routh-Hurwitz.

In [ ]:
num = [1.6, 78.2]
den = [-1.0, -37.1, -117.5, 0.0]
G = tf(num, den)

poles = np.roots(den)
zeros = np.roots(num)

real_poles = np.real(poles[np.isclose(np.imag(poles), 0)])
real_zeros = np.real(zeros[np.isclose(np.imag(zeros), 0)])

n = len(poles)
m_z = len(zeros)
r = n - m_z
centroid = (np.sum(poles) - np.sum(zeros)) / r
angles = [(2 * k + 1) * np.pi / r for k in range(r)]

# Intervalos reais do lugar das raízes
pts = np.sort(np.concatenate((real_poles, real_zeros)))
intervals = []
for i in range(len(pts) + 1):
    if i == 0:
        a = -1e6
        b_val = pts[0]
    elif i == len(pts):
        a = pts[-1]
        b_val = 1e6
    else:
        a = pts[i - 1]
        b_val = pts[i]
    mid = 0.5 * (a + b_val)
    count_right = np.sum(real_poles > mid) + np.sum(real_zeros > mid)
    if count_right % 2 == 1:
        intervals.append((a, b_val))

# Pontos de ruptura (dK/ds = 0)
s_sym = sp.symbols('s')
D_sym = -s_sym**3 - 37.1 * s_sym**2 - 117.5 * s_sym
N_sym = 1.6 * s_sym + 78.2
dD = sp.diff(D_sym, s_sym)
dN = sp.diff(N_sym, s_sym)

eq = sp.simplify(dD * N_sym - D_sym * dN)
sol = sp.solve(sp.Eq(eq, 0), s_sym)

sol_real = []
for item in sol:
    v = complex(sp.N(item))
    if abs(v.imag) < 1e-8:
        sol_real.append(float(v.real))
sol_real = sorted(set([round(x, 12) for x in sol_real]))

K_lower0 = -4359.25 / 18.84
K_lower = -1000

valid_breakpoints = []
break_K = []
for bp in sol_real:
    if not any(a < bp < b_val for (a, b_val) in intervals):
        continue
    D_val = float(sp.N(D_sym.subs(s_sym, bp)))
    N_val = float(sp.N(N_sym.subs(s_sym, bp)))
    if abs(N_val) < 1e-12:
        continue
    Kval = -D_val / N_val
    if np.isreal(Kval) and (Kval < 0) and (Kval > K_lower):
        valid_breakpoints.append(bp)
        break_K.append(float(Kval))

# Root locus numérico
num_p = np.array([1.6, 78.2])
den_p = np.array([-1.0, -37.1, -117.5, 0.0])
deg_den = len(den_p) - 1
deg_num = len(num_p) - 1
num_pad = np.pad(num_p, (deg_den - deg_num, 0))


def closed_loop_poles(K):
    coeff = den_p + K * num_pad
    if abs(coeff[0]) < 1e-12:
        coeff = np.trim_zeros(coeff, 'f')
    return np.roots(coeff)


Ks = np.linspace(K_lower * 0.999, -1e-3, 1200)

first = closed_loop_poles(Ks[0])
order = np.argsort(-np.real(first))
prev = list(first[order])
branches = [[z] for z in prev]

for K in Ks[1:]:
    cur = list(closed_loop_poles(K))
    assigned = [False] * len(cur)
    new = [None] * len(prev)

    for i, p in enumerate(prev):
        dists = [abs(p - c) if not assigned[j] else np.inf for j, c in enumerate(cur)]
        jmin = int(np.argmin(dists))
        assigned[jmin] = True
        new[i] = cur[jmin]

    for j, a in enumerate(assigned):
        if not a:
            for i in range(len(prev)):
                if new[i] is None:
                    new[i] = cur[j]
                    assigned[j] = True
                    break

    for i in range(len(prev)):
        branches[i].append(new[i])
    prev = new

branches = [np.array(b) for b in branches]

# Plot
fig, ax = plt.subplots(figsize=(11, 6))
ax.grid(True)
ax.set_title('Lugar das Raízes — Controlador de Rumo em Malha Fechada (Modelo Linear)')
ax.set_xlabel('Parte Real')
ax.set_ylabel('Parte Imaginária')

colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(branches)))
for col, c in zip(branches, colors):
    ax.plot(col.real, col.imag, '-', color=c, lw=2.0)

ax.plot(np.real(poles), np.imag(poles), 'x', markersize=10, color='cyan', label='polos')
ax.plot(
    np.real(zeros), np.imag(zeros), 'o', fillstyle='none',
    markersize=8, color='magenta', label='zeros',
)
ax.plot(np.real(centroid), 0, 's', color='yellow', markersize=8, label='centróide')

xext = np.linspace(-200, 200, 400)
for i, ang in enumerate(angles):
    lbl = 'Assíntotas' if i == 0 else None
    if np.isclose(np.mod(ang, np.pi), np.pi / 2):
        ax.plot([centroid.real, centroid.real], [-200, 200], '--', color='gray', label=lbl)
    else:
        ax.plot(xext, np.tan(ang) * (xext - centroid.real), '--', color='gray', label=lbl)

for a, b_val in intervals:
    left = a if a > -1e5 else -80
    right = b_val if b_val < 1e5 else 20
    ax.plot([left, right], [0, 0], color='red', lw=3)

for i, (bp, kv) in enumerate(zip(valid_breakpoints, break_K)):
    lbl = 'Ponto de ruptura' if i == 0 else None
    ax.plot(bp, 0, 'go', markersize=10, label=lbl)
    ax.annotate(
        f's = {bp:.2f}\n$K_p$ = {kv:.2f}',
        xy=(bp, 0),
        xytext=(bp, 6),
        ha='center',
        fontsize=9,
        color='white',
        bbox=dict(boxstyle='round,pad=0.4', fc='darkgreen', alpha=0.6),
    )

ax.text(
    0.45, 0.15,
    f'Faixa estável segundo Routh-Hurwitz: {K_lower0:.1f} < $K_p$ < 0',
    transform=ax.transAxes,
    fontsize=10,
    va='bottom',
    ha='right',
    bbox=dict(boxstyle='round,pad=0.3', fc='black', alpha=0.6),
)

ax.legend(loc='upper left')
ax.set_xlim(-60, 10)
ax.set_ylim(-40, 40)
plt.show()

### 5.3 Configuração do Controlador Proporcional

Ganho $K_p$ escolhido dentro da faixa estável de Routh-Hurwitz.
Matrizes do sistema controlável e condições iniciais.

In [ ]:
Kp = -1.0
Td = 1.0
a = np.deg2rad(15)  # amplitude do degrau de referência (rad)
r = a
phi_max = np.deg2rad(10.0)  # saturação do leme (rad)
phi_min = -phi_max

# Matrizes do sistema controlável — estados: [v, psi, psi_dot]
Ahc = np.zeros((3, 3))
Bhc = np.zeros((3, 1))

Ahc[0, 0] = -(
    (3 / 8) * U * math.sin(theta_m) * rho * 0.82 * CDc * D * L
    + 2 * rho * c * U * math.pi * b
    + 2 * 0.5 * rho * CDl * c * b * U
    + rho * L * D * U * math.pi
) / (m + m22)
Ahc[0, 2] = -U * (m + m11) / (m + m22)

Ahc[1, 2] = 1

Ahc[2, 0] = (
    2 * rho * c * b * U * math.pi * d
    + 2 * 0.5 * rho * CDl * c * b * U * d
    - 0.25 * rho * (L**2) * D * U * math.pi
) / (Jzz + m66)
Ahc[2, 2] = (
    -(omega_m * rho * 0.80 * CDc * D * (L**4))
    / (Jzz + m66)
    * (100 * (3 / 4) * (1 / 64))
)

Bhc[0, 0] = (2 * rho * c * (U**2) * math.pi * b) / (m + m33)
Bhc[2, 0] = -(2 * rho * c * b * (U**2) * math.pi * d) / (Jzz + m66)

Chc = np.array([[0, 1, 0]])

### 5.4 Funções de Dinâmica em Malha Fechada

In [ ]:
def LAUVhc_lin(t, X):
    X = X.reshape(-1, 1)
    y = Chc @ X

    u = Kp * (r - y)
    u_sat = np.clip(u, phi_min, phi_max)

    X_dot = Ahc @ X + Bhc @ u_sat
    return X_dot.flatten()


def LAUVhc_non_lin(t, X):
    u, v, psi, psi_dot = X

    u_ctrl = Kp * (r - psi)
    u_sat = np.clip(u_ctrl, phi_min, phi_max)

    beta = np.arctan2(v, u)
    phi = u_sat
    alpha = phi - beta

    Lf = Lift_foil(u, v, alpha)
    Rf = Drag_foil(u, v)
    Lc = Lift_hull(u, v, beta)

    A = np.array(
        [
            [m, 0, 0],
            [0, m + m22, 0],
            [0, 0, Jzz + m66],
        ]
    )
    b = np.array(
        [
            m * v * psi_dot,
            (
                2 * Lf * np.cos(beta)
                - 2 * Rf * np.sin(beta)
                - (m + m11) * u * psi_dot
                - (1 / 2) * rho * 0.82 * CDc * D * L * v * np.abs(v)
                - Lc * np.cos(beta)
            ),
            (
                -2 * (Lf * np.cos(beta) - Rf * np.sin(beta)) * d
                - 100
                * (1 / 64)
                * rho
                * 0.80
                * CDc
                * D
                * (L**4)
                * psi_dot
                * np.abs(psi_dot)
                - Lc * (L / 4) * np.cos(beta)
            ),
        ]
    )

    du_dt, dw_dt, dpsi_dot_dt = np.linalg.solve(A, b)
    dpsi_dt = psi_dot

    return np.hstack((du_dt, dw_dt, dpsi_dt, dpsi_dot_dt))

### 5.5 Simulação

In [ ]:
t_final = 30
t_eval = np.linspace(0, t_final, 2001)
X0_h = [U, 0, 0, 0]  # [u, v, psi, psi_dot]
X0c_h = [0, 0, 0]  # [v, psi, psi_dot]

solh_lin = solve_ivp(
    LAUVhc_lin, [0, t_final], X0c_h, t_eval=t_eval, atol=1e-8, rtol=1e-6
)
solh_non_lin = solve_ivp(
    LAUVhc_non_lin, [0, t_final], X0_h, t_eval=t_eval, atol=1e-8, rtol=1e-6
)

xs = solh_lin.y.T
ys = (Chc @ xs.T).flatten()
u_sat_lin = np.clip(Kp * (r - ys), phi_min, phi_max)

psi_non_lin = solh_non_lin.y[2]
psi_dot_non_lin = solh_non_lin.y[3]

u_non_lin = np.zeros(solh_non_lin.t.size)
for i in range(solh_non_lin.t.size):
    psi = solh_non_lin.y[2][i]
    u_non_lin[i] = np.clip(Kp * (r - psi), -phi_max, phi_max)

### 5.6 Resultados — Controle Horizontal

In [ ]:
# Ângulo de rumo
plt.figure(figsize=(10, 6))
plt.text(
    0.70, 0.40,
    f'$K_p$ = {Kp:.2f}',
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4'),
)
plt.plot(
    solh_lin.t, np.rad2deg(ys), color=ciano_neon, linestyle='--',
    label='$\psi$(t): Orientação Angular do LAUV — Modelo Linear',
)
plt.plot(
    solh_non_lin.t, np.rad2deg(psi_non_lin), color=ciano_neon, linestyle='-',
    label='$\psi$(t): Orientação Angular do LAUV — Modelo Não Linear',
)
plt.axhline(
    np.rad2deg(r), color='r', linestyle='--',
    label=f'ângulo de referência: $\psi$ = {np.rad2deg(r):.1f} graus',
)
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (graus)')
plt.title('Controlador Proporcional de Rumo (Yaw) do LAUV — Plano Horizontal')
plt.legend(loc='center right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Atuador
plt.figure(figsize=(10, 6))
plt.text(
    0.75, 0.40,
    f'$\phi_{{máx}}$ = {np.rad2deg(phi_max):.1f} graus',
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4'),
)
plt.plot(
    solh_lin.t, np.rad2deg(u_sat_lin), color=laranja_neon, linestyle='--',
    label='$\phi$(t): Ângulo do Leme — Modelo Linear',
)
plt.plot(
    solh_non_lin.t, np.rad2deg(u_non_lin), color=laranja_neon, linestyle='-',
    label='$\phi$(t): Ângulo do Leme — Modelo Não Linear',
)
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (graus)')
plt.title('Ângulo de Deflexão do Leme Saturado')
plt.legend(loc='center right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Plano de fase
plt.figure(figsize=(10, 6))
plt.plot(
    np.rad2deg(psi_non_lin), np.rad2deg(psi_dot_non_lin), color=vermelho_neon, linestyle='-',
    label='Trajetória do LAUV no Plano de Fase',
)
plt.xlabel('Ângulo (graus)')
plt.ylabel('Taxa Angular (graus/s)')
plt.title('Plano de Fase do Modelo Não Linear — Movimento no Plano Horizontal')
plt.legend(loc='upper right')
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()